Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 4/4: Оценка качества RAG → Урок 4/4: Итоговый анализ RAG-системы: от метрик к улучшениям]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/68295d2a-b17c-4564-8957-808c64a8e401/lessons/82867876-9a3d-429f-9504-4dd8974b2073/#5e864a69-6b41-4503-8685-a5eeb4e31f18 )


In [1]:
import json

with open("knowledge_base.json", encoding="utf-8") as f:
    data = json.load(f)
docs = data["docs"]            # Список словарей: 43 раздела, поля doc_id, title, text
queries = data["queries"]      # Список словарей: 30 запросов, поля query, kind,
                               #             relevant_docs, gold

print("Разделов:", len(docs), "| Запросов:", len(queries))
print(queries[0])

# Вывод:
# Разделов: 43 | Запросов: 30
# {'query': 'что означает код ошибки E-52', 'kind': 'exact',
#  'relevant_docs': [5], 'gold': 'перегрев процессора'}

Разделов: 43 | Запросов: 30
{'query': 'что означает код ошибки E-52', 'kind': 'exact', 'relevant_docs': [5], 'gold': 'перегрев процессора'}


In [2]:
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer


def naive_split(text, size=120):
    """Разрезает text на фрагменты по size символов без перекрытия.

    Возвращает список строк; границы предложений не учитываются.
    """
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks, chunk_doc = [], []                # Тексты чанков и раздел-источник
for d in docs:
    for piece in naive_split(d["text"]):
        chunks.append(piece)
        chunk_doc.append(d["doc_id"])     # Чанк помнит, из какого раздела вырезан

print("Чанков:", len(chunks))             # Чанков: 108

# Эмбеддер темы: multilingual-e5-base с префиксами query: и passage:
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
doc_vecs = embedder.encode(["passage: " + c for c in chunks],
                           normalize_embeddings=True, show_progress_bar=False)


def search(query, k=5):
    """Ищет для query ближайшие k чанков.

    Возвращает тексты чанков и идентификаторы разделов-источников.
    """
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    top = np.argsort(q_vec @ doc_vecs.T)[::-1][:k]
    return [chunks[i] for i in top], [chunk_doc[i] for i in top]


# Генерация: тот же локальный сервер и тот же заземляющий промпт
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def generate(messages, max_tokens=200):
    """Отправляет messages локальной модели и ограничивает ответ max_tokens.

    Возвращает очищенный текст ответа; temperature=0 — для воспроизводимости.
    """
    resp = client.chat.completions.create(
        model="qwen3.5", messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()


def rag_answer(query, k=5):
    """Запускает RAG для query по k найденным чанкам.

    Возвращает ответ, тексты чанков и идентификаторы разделов-источников.
    """
    ctx, got_docs = search(query, k)
    numbered = "\n".join(f"[{i}] {c}" for i, c in enumerate(ctx, 1))
    answer = generate([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content":
         f"Фрагменты базы знаний:\n{numbered}\n\nВопрос: {query}"}])
    return answer, ctx, got_docs

Чанков: 108


In [3]:
import re

JUDGE_FAITH = (
    "Ты — строгий эксперт по оценке ответов. Оцени, насколько ОТВЕТ опирается "
    "только на ФРАГМЕНТЫ. 5 — каждое утверждение ответа есть во фрагментах; "
    "1 — ответ содержит факты, которых во фрагментах нет. Ответ «в базе знаний "
    "ответа нет» считай полностью опирающимся на фрагменты (5). "
    "Верни только одну цифру от 1 до 5.")

JUDGE_RELEV = (
    "Ты — строгий эксперт по оценке ответов. Оцени, отвечает ли ОТВЕТ по "
    "существу на ВОПРОС. 5 — прямой и полный ответ на заданный вопрос; "
    "1 — ответ не по теме или отказ отвечать. Верни только одну цифру от 1 до 5.")

JUDGE_CORRECT = (
    "Ты — строгий эксперт. Сравни ОТВЕТ с ЭТАЛОННЫМ ФАКТОМ. Верни 1, если ответ "
    "передаёт эталонный факт (формулировка может отличаться), и 0, если ответ "
    "противоречит эталону, неполон или отказывается отвечать. "
    "Верни только одну цифру: 1 или 0.")


def judge(question, context, answer, gold):
    """Оценивает answer для question по context и эталонному gold.

    Возвращает faithfulness, answer relevance и бинарную correctness.
    """
    ctx = "\n".join(f"[{i}] {c}" for i, c in enumerate(context, 1))
    faith = generate([{"role": "system", "content": JUDGE_FAITH},
                      {"role": "user", "content":
                       f"ФРАГМЕНТЫ:\n{ctx}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    relev = generate([{"role": "system", "content": JUDGE_RELEV},
                      {"role": "user", "content":
                       f"ВОПРОС: {question}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    correct = generate([{"role": "system", "content": JUDGE_CORRECT},
                        {"role": "user", "content":
                         f"ВОПРОС: {question}\nЭТАЛОННЫЙ ФАКТ: {gold}\n\n"
                         f"ОТВЕТ: {answer}"}], max_tokens=8)
    return (int(re.search(r"[1-5]", faith).group()),
            int(re.search(r"[1-5]", relev).group()),
            int(re.search(r"[01]", correct).group()))

In [4]:
import pandas as pd
# Функции rag_answer(), judge() и список queries — из общего кода

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)


def precision_at_k(got_docs, relevant):
    """Доля слотов выдачи, занятых уникальными релевантными разделами."""
    # Пропуск 1а: каждый релевантный раздел учитывайте только один раз
    return len(set(relevant) & set(got_docs)) / len(got_docs)


def recall_at_k(got_docs, relevant):
    """Доля релевантных разделов, которые поиск донёс до промпта."""
    # Пропуск 1б: верните долю элементов relevant, попавших в got_docs
    return len(set(relevant) & set(got_docs)) / len(relevant)


def mrr(got_docs, relevant):
    """Обратный ранг первого релевантного раздела: 1 — первое место,
    0,5 — второе, 0 — релевантных в выдаче нет вовсе."""
    for rank, doc in enumerate(got_docs, start=1):
        # Пропуск 2: если doc релевантен, верните обратный ранг
        if doc in relevant:
            return 1/rank
    return 0.0


rows = []
for q in queries:
    answer, ctx, got_docs = rag_answer(q["query"], k=5)
    faith, relev, correct = judge(q["query"], ctx, answer, q["gold"])
    rows.append({
        "kind": q["kind"],
        "query": q["query"],
        "precision@5": precision_at_k(got_docs, q["relevant_docs"]),
        "recall@5": recall_at_k(got_docs, q["relevant_docs"]),
        "MRR": mrr(got_docs, q["relevant_docs"]),
        "faithfulness": faith,
        "answer relevance": relev,
        "correctness": correct,
        "answer": answer,
        "ctx": ctx,
    })

report = pd.DataFrame(rows)
METRICS = ["precision@5", "recall@5", "MRR", "faithfulness",
           "answer relevance", "correctness"]

# Пропуск 3: выведите средние по всему набору и разбивку по типам запросов
print("Сводный профиль:")
print(report[METRICS].mean().round(3))
print("\nПо типам запросов:")
print(report.groupby("kind")[METRICS].mean().round(3))

APIConnectionError: Connection error.

In [ ]:
gold_scores = []
for q in queries:
    ctx = [d["text"] for d in docs if d["doc_id"] in q["relevant_docs"]]
    numbered = "\n".join(f"[{i}] {c}" for i, c in enumerate(ctx, 1))
    answer = generate([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content":
         f"Фрагменты базы знаний:\n{numbered}\n\nВопрос: {q['query']}"}])
    gold_scores.append(judge(q["query"], ctx, answer, q["gold"])[2])

print("correctness на золотом контексте:", round(np.mean(gold_scores), 3))
# correctness на золотом контексте: 0.967

In [ ]:
import re
import time
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder

# Что берём из задания с профилем качества: report — профиль «до», а также написанные вами
# precision_at_k(), recall_at_k() и mrr()
# Что берём из общего кода: docs, queries, embedder, chunks, chunk_doc,
#                           doc_vecs, generate(), judge(), SYSTEM

METRICS = ["precision@5", "recall@5", "MRR", "faithfulness",
           "answer relevance", "correctness"]


def evaluate(search_fn):
    """Прогоняет 30 запросов через переданную функцию поиска.

    Возвращает такой же отчёт, как в задании с профилем качества: метрики поиска и генерации
    на каждый запрос. Функция search_fn принимает запрос и возвращает
    тексты чанков и идентификаторы разделов-источников.
    """
    rows = []
    for q in queries:
        ctx, got_docs = search_fn(q["query"])
        numbered = "\n".join(f"[{i}] {c}" for i, c in enumerate(ctx, 1))
        answer = generate([
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content":
             f"Фрагменты базы знаний:\n{numbered}\n\nВопрос: {q['query']}"}])
        faith, relev, correct = judge(q["query"], ctx, answer, q["gold"])
        rows.append({
            "kind": q["kind"], "query": q["query"],
            "precision@5": precision_at_k(got_docs, q["relevant_docs"]),
            "recall@5": recall_at_k(got_docs, q["relevant_docs"]),
            "MRR": mrr(got_docs, q["relevant_docs"]),
            "faithfulness": faith, "answer relevance": relev,
            "correctness": correct, "answer": answer, "ctx": ctx,
        })
    return pd.DataFrame(rows)


# ---------- Кандидат А: перенарезка по границам предложений ----------
def sentence_split(text, size=120, overlap_sentences=1):
    """Собирает чанки до size символов по границам предложений.

    Переносит в следующий чанк до overlap_sentences целых предложений
    и возвращает список строк.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    parts, buf = [], []
    for sentence in sentences:
        candidate = " ".join([*buf, sentence])
        if not buf or len(candidate) <= size:
            buf.append(sentence)
        else:
            # Пропуск 1: сохраните чанк и перенесите целые предложения
            #            в начало следующего чанка
            parts.append(" ".join(buf))
            # Перекрытие: последнее целое предложение уезжает в новый чанк
            buf = (buf[-overlap_sentences:] + [sentence]
                   if overlap_sentences else [sentence])
            # Если перекрытие вместе с новым предложением не влезло в окно,
            # выбрасываем самое старое предложение
            while len(" ".join(buf)) > size and len(buf) > 1:
                buf.pop(0)
    if buf:
        parts.append(" ".join(buf))
    return [p for p in parts if p]


chunks_a, chunk_doc_a = [], []
for d in docs:
    for piece in sentence_split(d["text"]):
        chunks_a.append(piece)
        chunk_doc_a.append(d["doc_id"])
print("Чанков после перенарезки:", len(chunks_a))

vecs_a = embedder.encode(["passage: " + c for c in chunks_a],
                         normalize_embeddings=True, show_progress_bar=False)


def search_a(query, k=5):
    """Плотный поиск по перенарезанной базе."""
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    top = np.argsort(q_vec @ vecs_a.T)[::-1][:k]
    return [chunks_a[i] for i in top], [chunk_doc_a[i] for i in top]


report_a = evaluate(search_a)

# ---------- Кандидат Б: реранкер поверх исходной, битой нарезки ----------
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda")


def search_b(query, k=5, n_cand=20):
    """Плотный поиск отбирает n_cand кандидатов, cross-encoder — k лучших."""
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    cand = np.argsort(q_vec @ doc_vecs.T)[::-1][:n_cand]
    # Пропуск 2: оцените пары (запрос, чанк) реранкером и оставьте k лучших.
    #            reranker.predict([(query, текст_чанка), ...]) вернёт массив
    #            оценок в том же порядке, что и переданные пары
    scores = reranker.predict([(query, chunks[i]) for i in cand])
    top = [cand[i] for i in np.argsort(scores)[::-1][:k]]
    return [chunks[i] for i in top], [chunk_doc[i] for i in top]


report_b = evaluate(search_b)


# ---------- Сравнение: доказано или нет ----------
def bootstrap_ci(before, after, n_boot=10000):
    """Строит 95-процентный интервал разницы before и after.

    Использует n_boot парных выборок и возвращает разницу и границы интервала.
    """
    rng = np.random.default_rng(42)
    diff = np.array(after, dtype=float) - np.array(before, dtype=float)
    idx = rng.integers(0, len(diff), size=(n_boot, len(diff)))
    # Пропуск 3: посчитайте среднюю разницу на каждой из n_boot выборок
    #            и верните её среднее плюс 2,5-й и 97,5-й процентили
    means = diff[idx].mean(axis=1)          # Средняя разница на каждой выборке
    return diff.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


def summarize(before, after, title):
    """Печатает таблицу «до / после / разница», интервалы и счёт починок."""
    print(f"\n=== {title} ===")
    # Пропуск 4: соберите таблицу со столбцами «до» и «после» — средними
    #            по METRICS в before и after — и добавьте столбец «разница»
    compare = pd.DataFrame({
        "до": before[METRICS].mean(),
        "после": after[METRICS].mean()
    }).round(3)
    compare["разница"] = (compare["после"] - compare["до"]).round(3)
    print(compare)

    for metric in ["recall@5", "correctness"]:
        d, lo, hi = bootstrap_ci(before[metric], after[metric])
        if lo > 0:
            verdict = "значимое улучшение"
        elif hi < 0:
            verdict = "значимое ухудшение"
        else:
            verdict = "изменение не доказано"
        print(f"{metric:12} Δ={d:+.3f}  95% ДИ [{lo:+.3f}; {hi:+.3f}]  {verdict}")

    # Пропуск 5: посчитайте, на скольких запросах correctness поднялась
    #            с 0 до 1 (починились) и на скольких упала с 1 до 0 (сломались)
    fixed = ((before["correctness"] == 0) & (after["correctness"] == 1)).sum()
    broken = ((before["correctness"] == 1) & (after["correctness"] == 0)).sum()
    print(f"Починилось: {fixed}, сломалось: {broken}")


summarize(report, report_a, "А: перенарезка по предложениям")
summarize(report, report_b, "Б: реранкер bge-reranker-v2-m3 (20 → 5)")


# ---------- Цена: во что обходится каждый кандидат ----------
def latency_ms(search_fn, repeats=3):
    """Медианное время одного поиска в миллисекундах."""
    times = []
    for _ in range(repeats):
        for q in queries:
            start = time.perf_counter()
            search_fn(q["query"])
            times.append((time.perf_counter() - start) * 1000)
    return np.median(times)


print(f"\nПоиск без реранкера: {latency_ms(search_a):.0f} мс")
print(f"Поиск с реранкером:  {latency_ms(search_b):.0f} мс")